# 🎓 עוזר ניתוח סטטיסטי לסטודנטים

**מה הכלי הזה עושה?**
- מאפשר לך להעלות קובץ Excel עם הנתונים שלך
- מנחה אותך צעד אחר צעד בניתוח סטטיסטי
- עוזר לזהות משתנים תלויים ובלתי תלויים
- ממליץ על סוג הניתוח המתאים
- מריץ את הניתוחים ומציג תוצאות

**שלבים:**
1. התקנות וחיבור API
2. העלאת קובץ נתונים
3. סטטיסטיקה תיאורית (EDA)
4. שיחה עם העוזר לתכנון הניתוח
5. הרצת ניתוחים

## 0. התקנות והגדרות

In [ ]:
!pip install -q google-generativeai openpyxl scipy statsmodels pingouin scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
import json
import textwrap
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('✅ כל הספריות נטענו בהצלחה')

In [ ]:
import google.generativeai as genai

# --- הגדרת API Key ---
# אפשרות 1: מ-Colab Secrets (מומלץ)
try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    # אפשרות 2: הזנה ידנית
    API_KEY = input('הכניסי את ה-Gemini API Key שלך: ')

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash')
print('✅ Gemini מחובר')

## 1. העלאת קובץ נתונים

In [ ]:
# --- העלאת קובץ ---
# אפשרות 1: העלאה מהמחשב (Colab)
try:
    from google.colab import files
    print('📁 בחרי קובץ Excel להעלאה:')
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
except ImportError:
    # אפשרות 2: נתיב ידני (Jupyter רגיל)
    FILE_PATH = input('הכניסי נתיב לקובץ Excel: ')

# קריאת הקובץ
xl = pd.ExcelFile(FILE_PATH, engine='openpyxl')
print(f'\nשמות הגיליונות בקובץ: {xl.sheet_names}')

if len(xl.sheet_names) > 1:
    sheet = input(f'איזה גיליון לטעון? ({xl.sheet_names}): ') or xl.sheet_names[0]
else:
    sheet = xl.sheet_names[0]

df = pd.read_excel(FILE_PATH, sheet_name=sheet, engine='openpyxl')
print(f'\n✅ נטענו {len(df)} שורות ו-{len(df.columns)} עמודות')
print(f'\nשמות העמודות:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col} ({df[col].dtype})')

## 2. סטטיסטיקה תיאורית (EDA)

In [ ]:
def run_eda(df):
    """הרצת ניתוח תיאורי מקיף על הנתונים."""
    eda_report = {}

    # --- מידע כללי ---
    print('=' * 60)
    print('📊 סיכום כללי של הנתונים')
    print('=' * 60)
    print(f'מספר שורות: {len(df)}')
    print(f'מספר עמודות: {len(df.columns)}')
    print()

    # --- סיווג עמודות ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()

    # עמודות נומריות עם מעט ערכים ייחודיים -> קטגוריאליות
    pseudo_cat = [c for c in numeric_cols if df[c].nunique() <= 10]
    if pseudo_cat:
        print(f'⚠️ עמודות נומריות עם עד 10 ערכים (ייתכן שהן קטגוריאליות):')
        for c in pseudo_cat:
            print(f'   {c}: ערכים = {sorted(df[c].dropna().unique())}')
        print()

    eda_report['numeric_cols'] = numeric_cols
    eda_report['categorical_cols'] = categorical_cols
    eda_report['datetime_cols'] = datetime_cols
    eda_report['pseudo_categorical'] = pseudo_cat

    # --- ערכים חסרים ---
    print('=' * 60)
    print('🔍 ערכים חסרים (Missing Values)')
    print('=' * 60)
    missing = pd.DataFrame({
        'חסרים': df.isnull().sum(),
        'אחוז_חסרים': (df.isnull().sum() / len(df) * 100).round(1)
    }).sort_values('אחוז_חסרים', ascending=False)
    missing = missing[missing['חסרים'] > 0]
    if len(missing) > 0:
        display(missing)
    else:
        print('אין ערכים חסרים!')
    print()
    eda_report['missing'] = missing.to_dict()

    # --- משתנים נומריים ---
    if numeric_cols:
        print('=' * 60)
        print('📈 סטטיסטיקה תיאורית - משתנים נומריים')
        print('=' * 60)
        desc = df[numeric_cols].describe().round(2).T
        desc['median'] = df[numeric_cols].median().round(2)
        desc['skew'] = df[numeric_cols].skew().round(2)
        display(desc)
        print()
        eda_report['numeric_summary'] = desc.to_dict()

    # --- משתנים קטגוריאליים ---
    if categorical_cols:
        print('=' * 60)
        print('📋 סטטיסטיקה תיאורית - משתנים קטגוריאליים')
        print('=' * 60)
        cat_summaries = {}
        for col in categorical_cols:
            vc = df[col].value_counts()
            pct = (df[col].value_counts(normalize=True) * 100).round(1)
            summary = pd.DataFrame({'ספירה': vc, 'אחוז': pct})
            print(f'\n--- {col} ({df[col].nunique()} ערכים ייחודיים) ---')
            display(summary.head(15))
            cat_summaries[col] = summary.head(15).to_dict()
        eda_report['categorical_summary'] = cat_summaries
    print()

    return eda_report

eda_report = run_eda(df)

In [ ]:
def plot_distributions(df, max_plots=12):
    """הצגת התפלגויות לכל המשתנים הנומריים."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        print('אין משתנים נומריים להצגה')
        return

    cols_to_plot = numeric_cols[:max_plots]
    n = len(cols_to_plot)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(cols_to_plot):
        data = df[col].dropna()
        axes[i].hist(data, bins=30, edgecolor='white', alpha=0.7)
        axes[i].set_title(col, fontsize=11)
        axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.1f}')
        axes[i].axvline(data.median(), color='green', linestyle='-', label=f'median={data.median():.1f}')
        axes[i].legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.suptitle('התפלגויות משתנים נומריים', y=1.02, fontsize=14)
    plt.show()

plot_distributions(df)

In [ ]:
def plot_correlation_matrix(df):
    """מטריצת מתאמים למשתנים נומריים."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) < 2:
        print('צריך לפחות 2 משתנים נומריים למטריצת מתאמים')
        return

    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), max(6, len(numeric_cols) * 0.6)))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax)
    plt.title('מטריצת מתאמים (Pearson)')
    plt.tight_layout()
    plt.show()

plot_correlation_matrix(df)

## 3. שיחה עם העוזר הסטטיסטי

עכשיו נתחיל שיחה עם העוזר. הוא:
- יעזור לך לזהות משתנה תלוי ובלתי תלויים
- ימליץ על סוג הניתוח
- יכתוב ויריץ קוד ניתוח

**כתבי `quit` או `יציאה` לסיום השיחה.**

In [ ]:
# --- פונקציות ניתוח שהצ'אט יכול להפעיל ---

def analyze_normality(df, col):
    """בדיקת נורמליות למשתנה."""
    data = df[col].dropna()
    stat_shapiro, p_shapiro = stats.shapiro(data[:5000])  # shapiro מוגבל
    stat_ks, p_ks = stats.kstest(data, 'norm', args=(data.mean(), data.std()))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(data, bins=30, density=True, alpha=0.7, edgecolor='white')
    x = np.linspace(data.min(), data.max(), 100)
    axes[0].plot(x, stats.norm.pdf(x, data.mean(), data.std()), 'r-', lw=2)
    axes[0].set_title(f'התפלגות {col}')
    stats.probplot(data, dist='norm', plot=axes[1])
    axes[1].set_title(f'Q-Q Plot - {col}')
    plt.tight_layout()
    plt.show()

    result = f"""בדיקת נורמליות עבור {col}:
  Shapiro-Wilk: W={stat_shapiro:.4f}, p={p_shapiro:.4f} {'✅ נורמלי' if p_shapiro > 0.05 else '❌ לא נורמלי'}
  Kolmogorov-Smirnov: D={stat_ks:.4f}, p={p_ks:.4f} {'✅ נורמלי' if p_ks > 0.05 else '❌ לא נורמלי'}
  N={len(data)}, Mean={data.mean():.2f}, Std={data.std():.2f}, Skewness={data.skew():.2f}"""
    print(result)
    return result


def compare_two_groups(df, numeric_col, group_col):
    """השוואת שתי קבוצות."""
    groups = df[group_col].dropna().unique()
    if len(groups) != 2:
        return f'צריך בדיוק 2 קבוצות, נמצאו {len(groups)}: {groups}'

    g1 = df[df[group_col] == groups[0]][numeric_col].dropna()
    g2 = df[df[group_col] == groups[1]][numeric_col].dropna()

    # t-test
    t_stat, t_p = stats.ttest_ind(g1, g2)
    # Mann-Whitney
    u_stat, u_p = stats.mannwhitneyu(g1, g2, alternative='two-sided')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(g1, bins=20, alpha=0.6, label=f'{groups[0]} (n={len(g1)})')
    axes[0].hist(g2, bins=20, alpha=0.6, label=f'{groups[1]} (n={len(g2)})')
    axes[0].legend()
    axes[0].set_title(f'התפלגות {numeric_col} לפי {group_col}')
    sns.boxplot(data=df, x=group_col, y=numeric_col, ax=axes[1])
    axes[1].set_title(f'Box Plot - {numeric_col} לפי {group_col}')
    plt.tight_layout()
    plt.show()

    result = f"""השוואת {numeric_col} בין קבוצות {group_col}:
  {groups[0]}: n={len(g1)}, mean={g1.mean():.2f}, std={g1.std():.2f}
  {groups[1]}: n={len(g2)}, mean={g2.mean():.2f}, std={g2.std():.2f}
  Independent t-test: t={t_stat:.3f}, p={t_p:.4f} {'*' if t_p < 0.05 else 'NS'}
  Mann-Whitney U: U={u_stat:.1f}, p={u_p:.4f} {'*' if u_p < 0.05 else 'NS'}"""
    print(result)
    return result


def compare_multiple_groups(df, numeric_col, group_col):
    """השוואת מספר קבוצות - ANOVA / Kruskal-Wallis."""
    groups_data = [group[numeric_col].dropna() for _, group in df.groupby(group_col)]
    group_names = [name for name, _ in df.groupby(group_col)]

    f_stat, f_p = stats.f_oneway(*groups_data)
    h_stat, h_p = stats.kruskal(*groups_data)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=df, x=group_col, y=numeric_col, ax=ax)
    ax.set_title(f'{numeric_col} לפי {group_col}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    result = f"""השוואת {numeric_col} בין קבוצות {group_col} ({len(group_names)} קבוצות):
  One-Way ANOVA: F={f_stat:.3f}, p={f_p:.4f} {'*' if f_p < 0.05 else 'NS'}
  Kruskal-Wallis: H={h_stat:.3f}, p={h_p:.4f} {'*' if h_p < 0.05 else 'NS'}"""

    # Post-hoc if significant
    if f_p < 0.05:
        try:
            import pingouin as pg
            posthoc = pg.pairwise_tukey(data=df.dropna(subset=[numeric_col, group_col]),
                                        dv=numeric_col, between=group_col)
            result += '\n\n  Post-hoc Tukey HSD:'
            print(result)
            display(posthoc[['A', 'B', 'diff', 'p-tukey']].round(4))
        except Exception:
            print(result)
    else:
        print(result)
    return result


def run_correlation(df, col1, col2):
    """מתאם בין שני משתנים."""
    clean = df[[col1, col2]].dropna()
    r_pearson, p_pearson = stats.pearsonr(clean[col1], clean[col2])
    r_spearman, p_spearman = stats.spearmanr(clean[col1], clean[col2])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(clean[col1], clean[col2], alpha=0.5)
    z = np.polyfit(clean[col1], clean[col2], 1)
    p = np.poly1d(z)
    ax.plot(sorted(clean[col1]), p(sorted(clean[col1])), 'r--', lw=2)
    ax.set_xlabel(col1)
    ax.set_ylabel(col2)
    ax.set_title(f'מתאם בין {col1} ו-{col2}')
    plt.tight_layout()
    plt.show()

    result = f"""מתאם בין {col1} ו-{col2} (n={len(clean)}):
  Pearson:  r={r_pearson:.3f}, p={p_pearson:.4f} {'*' if p_pearson < 0.05 else 'NS'}
  Spearman: r={r_spearman:.3f}, p={p_spearman:.4f} {'*' if p_spearman < 0.05 else 'NS'}"""
    print(result)
    return result


def run_chi_square(df, col1, col2):
    """מבחן חי-בריבוע לשני משתנים קטגוריאליים."""
    ct = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, expected = stats.chi2_contingency(ct)

    print('טבלת שכיחויות (Crosstab):')
    display(ct)

    # Cramér's V
    n = ct.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

    fig, ax = plt.subplots(figsize=(8, 5))
    ct.plot(kind='bar', ax=ax)
    ax.set_title(f'{col1} vs {col2}')
    plt.tight_layout()
    plt.show()

    result = f"""Chi-Square test: {col1} × {col2}
  χ²={chi2:.3f}, df={dof}, p={p:.4f} {'*' if p < 0.05 else 'NS'}
  Cramér's V = {cramers_v:.3f}"""
    print(result)
    return result


def run_linear_regression(df, dependent, independents):
    """רגרסיה ליניארית."""
    clean = df[[dependent] + independents].dropna()
    X = sm.add_constant(clean[independents])
    y = clean[dependent]
    result = sm.OLS(y, X).fit()

    print(result.summary())

    # Residuals plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(result.fittedvalues, result.resid, alpha=0.5)
    axes[0].axhline(y=0, color='r', linestyle='--')
    axes[0].set_xlabel('Fitted Values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Fitted')
    stats.probplot(result.resid, dist='norm', plot=axes[1])
    axes[1].set_title('Q-Q Plot of Residuals')
    plt.tight_layout()
    plt.show()

    return str(result.summary())


def run_logistic_regression(df, dependent, independents):
    """רגרסיה לוגיסטית."""
    clean = df[[dependent] + independents].dropna()
    X = sm.add_constant(clean[independents])
    y = clean[dependent]
    result = sm.Logit(y, X).fit(disp=0)

    print(result.summary())

    # Odds ratios
    odds = pd.DataFrame({
        'coef': result.params,
        'OR': np.exp(result.params),
        'OR_CI_low': np.exp(result.conf_int()[0]),
        'OR_CI_high': np.exp(result.conf_int()[1]),
        'p_value': result.pvalues
    }).round(4)
    print('\nOdds Ratios:')
    display(odds)

    return str(result.summary())


# מילון הפונקציות הזמינות
AVAILABLE_FUNCTIONS = {
    'analyze_normality': analyze_normality,
    'compare_two_groups': compare_two_groups,
    'compare_multiple_groups': compare_multiple_groups,
    'run_correlation': run_correlation,
    'run_chi_square': run_chi_square,
    'run_linear_regression': run_linear_regression,
    'run_logistic_regression': run_logistic_regression,
}

print('✅ פונקציות הניתוח מוכנות')

In [ ]:
# --- הגדרת הצ'אט הסטטיסטי ---

def build_data_context(df, eda_report):
    """בניית תיאור הנתונים עבור ה-LLM."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    context = f"""הנתונים מכילים {len(df)} שורות ו-{len(df.columns)} עמודות.

משתנים נומריים ({len(numeric_cols)}):
"""
    for col in numeric_cols:
        data = df[col].dropna()
        context += f"  - {col}: n={len(data)}, mean={data.mean():.2f}, std={data.std():.2f}, min={data.min():.2f}, max={data.max():.2f}\n"

    context += f"\nמשתנים קטגוריאליים ({len(categorical_cols)}):\n"
    for col in categorical_cols:
        vals = df[col].value_counts().head(5)
        context += f"  - {col}: {df[col].nunique()} ערכים. נפוצים: {dict(vals)}\n"

    missing_cols = df.columns[df.isnull().any()].tolist()
    if missing_cols:
        context += f"\nעמודות עם ערכים חסרים: "
        for col in missing_cols:
            context += f"{col} ({df[col].isnull().sum()}/{len(df)}), "

    return context


SYSTEM_PROMPT = """אתה עוזר סטטיסטי מומחה שמנחה סטודנטים בניתוח נתונים. דבר בעברית.

תפקידך:
1. לעזור לסטודנט להבין את הנתונים שלו
2. לשאול שאלות כדי להבין מה שאלת המחקר
3. לעזור לזהות את המשתנה התלוי (dependent) והבלתי תלויים (independent)
4. להמליץ על סוג הניתוח המתאים בהתאם לסוגי המשתנים
5. להריץ ניתוחים ולהסביר את התוצאות

כללים לבחירת ניתוח:
- 2 משתנים נומריים → מתאם (Pearson/Spearman)
- נומרי תלוי + קטגוריאלי בלתי תלוי (2 קבוצות) → t-test / Mann-Whitney
- נומרי תלוי + קטגוריאלי בלתי תלוי (3+ קבוצות) → ANOVA / Kruskal-Wallis
- 2 משתנים קטגוריאליים → Chi-square
- נומרי תלוי + מספר בלתי תלויים → רגרסיה ליניארית
- בינארי תלוי + מספר בלתי תלויים → רגרסיה לוגיסטית
- תמיד בדוק נורמליות לפני מבחנים פרמטריים

כשאתה רוצה להריץ ניתוח, החזר JSON בפורמט הבא (בתוך ```json ... ```):
```json
{"function": "שם_פונקציה", "args": {"arg1": "value1"}}
```

פונקציות זמינות:
- analyze_normality(col) - בדיקת נורמליות
- compare_two_groups(numeric_col, group_col) - השוואת 2 קבוצות
- compare_multiple_groups(numeric_col, group_col) - השוואת 3+ קבוצות
- run_correlation(col1, col2) - מתאם בין 2 משתנים נומריים
- run_chi_square(col1, col2) - חי-בריבוע ל-2 קטגוריאליים
- run_linear_regression(dependent, independents) - רגרסיה ליניארית (independents=רשימה)
- run_logistic_regression(dependent, independents) - רגרסיה לוגיסטית (independents=רשימה)

הנחיות חשובות:
- התחל תמיד בשאלה "מה שאלת המחקר שלך?"
- הסבר כל צעד בשפה פשוטה
- הסבר למה בחרת את הניתוח הזה
- הריץ ניתוח אחד בכל פעם
- אחרי כל תוצאה, הסבר מה היא אומרת
- אם הסטודנט לא בטוח, הצע אפשרויות
"""


def extract_function_call(text):
    """חילוץ קריאת פונקציה מתשובת ה-LLM."""
    import re
    match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None


def execute_analysis(func_call, df):
    """הרצת פונקציית ניתוח."""
    func_name = func_call.get('function')
    args = func_call.get('args', {})

    if func_name not in AVAILABLE_FUNCTIONS:
        return f'פונקציה לא מוכרת: {func_name}'

    func = AVAILABLE_FUNCTIONS[func_name]
    try:
        return func(df, **args)
    except Exception as e:
        return f'שגיאה בהרצת {func_name}: {str(e)}'


print('✅ מנוע הצ\'אט מוכן')

In [ ]:
# --- לולאת השיחה הראשית ---

data_context = build_data_context(df, eda_report)

chat = model.start_chat(history=[
    {'role': 'user', 'parts': [f'הנה תיאור הנתונים שלי:\n{data_context}']},
    {'role': 'model', 'parts': ['תודה! קיבלתי את הנתונים. אני מוכן לעזור לך עם הניתוח הסטטיסטי.']}
])

print('=' * 60)
print('🤖 עוזר סטטיסטי - התחלת שיחה')
print('=' * 60)

# שאלת פתיחה
opening = chat.send_message(
    'הסטודנט העלה את הנתונים. התחל בברכה קצרה, תאר את הנתונים בקצרה, ושאל מה שאלת המחקר.'
)
print(f'\n🤖 עוזר: {opening.text}\n')

while True:
    user_input = input('📝 את/ה: ')
    if user_input.strip().lower() in ['quit', 'exit', 'יציאה', 'סיום', 'q']:
        print('\n👋 תודה! בהצלחה עם המחקר.')
        break

    response = chat.send_message(user_input)
    response_text = response.text

    # בדיקה אם יש קריאת פונקציה
    func_call = extract_function_call(response_text)

    if func_call:
        # הצגת ההסבר (הטקסט לפני ה-JSON)
        explanation = response_text.split('```json')[0].strip()
        if explanation:
            print(f'\n🤖 עוזר: {explanation}')

        print(f'\n⚙️ מריץ: {func_call["function"]}...')
        print('-' * 40)

        result = execute_analysis(func_call, df)

        print('-' * 40)

        # שליחת התוצאות חזרה ל-LLM לפירוש
        interpretation = chat.send_message(
            f'הנה תוצאות הניתוח:\n{result}\n\nהסבר לסטודנט את התוצאות בשפה פשוטה, ושאל מה הצעד הבא.'
        )
        print(f'\n🤖 עוזר: {interpretation.text}\n')
    else:
        print(f'\n🤖 עוזר: {response_text}\n')

## 4. הרצת ניתוחים ידנית (אופציונלי)

אם את/ה רוצה להריץ ניתוח ספציפי ישירות, ללא הצ'אט:

In [ ]:
# --- דוגמאות לשימוש ישיר ---
# הסירי את ה-# מהשורה הרלוונטית והחליפי את שמות העמודות

# בדיקת נורמליות:
# analyze_normality(df, 'שם_עמודה')

# השוואת 2 קבוצות:
# compare_two_groups(df, 'משתנה_נומרי', 'משתנה_קבוצה')

# מתאם:
# run_correlation(df, 'עמודה1', 'עמודה2')

# חי-בריבוע:
# run_chi_square(df, 'קטגוריאלי1', 'קטגוריאלי2')

# רגרסיה ליניארית:
# run_linear_regression(df, 'משתנה_תלוי', ['בלתי_תלוי_1', 'בלתי_תלוי_2'])

# רגרסיה לוגיסטית:
# run_logistic_regression(df, 'משתנה_בינארי', ['בלתי_תלוי_1', 'בלתי_תלוי_2'])